In [1]:
from functools import cache

# 1) One-Dimensional DP

**The gist.** Solve a big problem by solving smaller versions once, writing the answers down, and building the big answer from them. It pays off when the same small questions keep coming up.

> **Think of it like…** counting the ways to reach each step of a staircase: once "step 5 = 8 ways" is written on the wall, nobody recounts it — steps 6 and 7 just read the number.

Write these four down before any code:

| Step | Question | Example (House Robber) |
|---|---|---|
| **State** | what does `dp[i]` mean, in words? | best loot from houses `0..i` |
| **Transition** | how is it built from smaller states? | `dp[i] = max(dp[i-1], dp[i-2] + nums[i])` |
| **Base case** | smallest states you already know | `dp[0] = nums[0]` |
| **Answer** | which state holds the result? | `dp[n-1]` |

Two ways to write it: **top-down** (recursion + `@cache` — closest to how you think, start here) or **bottom-up** (a table filled in order — lets you **shrink space** when each row only needs the previous one).

| Family | Cue | State |
|---|---|---|
| 1D linear | "ways to reach / best up to i" | `dp[i]` |
| Knapsack | pick items under a capacity / target | `dp[capacity]` |
| Two sequences | compare / align two strings | `dp[i][j]` over prefixes |
| Grid | paths moving right/down | `dp[r][c]` |
| Intervals | split / merge a range, games | `dp[l][r]` |
| State machine | holding / not holding / cooldown | `dp[i][state]` |

**Spot it:** "number of ways", "minimum / maximum", "can you reach / build", and the obvious greedy choice can be wrong.

**Classic bugs:** a state that isn't written down in words (then the transition drifts) · wrong base case (`dp[0]` for "empty" is often 1 for counting) · off-by-one between `dp` of length `n + 1` and the input.

## 1.1 Fibonacci-style recurrences

`dp[i]` depends on a fixed number of earlier states (`i − 1`, `i − 2`). Write the recurrence, then keep only the last few values in variables — O(1) space.

### 1.1.1 [70. Climbing Stairs](https://leetcode.com/problems/climbing-stairs/) — ★★★ · Easy

**Problem.** Climb `n` steps taking 1 or 2 at a time; how many distinct ways?  
**Example.** `n = 3` → `3` (1+1+1, 1+2, 2+1)  
**Spot it.** Ways to reach step `i` = ways to reach `i−1` + ways to reach `i−2` → Fibonacci.

**Ask first**
- Only 1 or 2 steps at a time? → Yes. With steps from a set, sum over that set.

> 💡 **Idea.** To land on step n, your last move was from n − 1 or n − 2. So ways(n) = ways(n − 1) + ways(n − 2) — Fibonacci.

**Brute force:** plain recursion, O(2ⁿ) — the same steps get recounted.

**Steps**
1. Start with ways(0) = 1 and ways(1) = 1.
2. Roll forward: `prev, cur = cur, prev + cur`.
3. After n − 1 rolls, `cur` is the answer.

**Cost:** Time O(n) · Space O(1)

In [ ]:
def climb_stairs(n):
    """Number of 1/2-step ways to climb n stairs."""
    # dp[i] = number of ways to reach step i (steps 0 and 1 each have 1 way)
    dp = [1] * (n + 1)
    # O(1) space: prev, cur = 1, 1
    for i in range(2, n + 1):
        # arrive from one step below or two steps below
        dp[i] = dp[i - 1] + dp[i - 2]
        # O(1) space: prev, cur = cur, prev + cur
    return dp[n]
    # O(1) space: return cur


assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(45) == 1836311903
print(climb_stairs(5))

8


### 1.1.2 [746. Min Cost Climbing Stairs](https://leetcode.com/problems/min-cost-climbing-stairs/) — ★★ · Easy

**Problem.** Pay `cost[i]` to step on stair `i`, then climb 1 or 2; start at index 0 or 1. Minimum cost to get past the top?  
**Example.** `cost = [10, 15, 20]` → `15` (step on the 15, then jump past the top)  
**Spot it.** `dp[i]` = cheapest cost to stand on step `i`.

**Ask first**
- Can you start on step 0 or 1 for free? → Yes.

> 💡 **Idea.** Cost to stand on step i = its cost + the cheaper of the two steps below. You can finish from either of the last two steps.

**Brute force:** plain recursion, O(2ⁿ).

**Steps**
1. `dp[0] = cost[0]`, `dp[1] = cost[1]`.
2. `dp[i] = cost[i] + min(dp[i − 1], dp[i − 2])`.
3. The top is reached from either of the last two steps: `min(dp[n − 1], dp[n − 2])`.

**Cost:** Time O(n) · Space O(1) with two variables

In [ ]:
def min_cost_climbing_stairs(cost):
    """Cheapest way past the last stair."""
    n = len(cost)
    # dp[i] = cheapest cost to stand on step i (you pay cost[i] to step on it)
    dp = [0] * n
    dp[0], dp[1] = cost[0], cost[1]
    # O(1) space: a, b = cost[0], cost[1]
    for i in range(2, n):
        # arrive from one or two steps below, whichever is cheaper
        dp[i] = cost[i] + min(dp[i - 1], dp[i - 2])
        # O(1) space: a, b = b, cost[i] + min(a, b)
    # jump to the top from either of the last two steps
    return min(dp[n - 1], dp[n - 2])
    # O(1) space: return min(a, b)


assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
print(min_cost_climbing_stairs([10, 15, 20]))

15


### 1.1.3 [198. House Robber](https://leetcode.com/problems/house-robber/) — ★★★ · Medium

**Problem.** Rob houses in a row without robbing two adjacent ones; maximise loot.  
**Example.** `[1, 2, 3, 1]` → `4` (rob houses 1 and 3: 1 + 3)  
**Spot it.** Take-or-skip at each index → `dp[i] = max(dp[i−1], dp[i−2] + nums[i])`.

**Ask first**
- Can values be 0? → Yes, non-negative — so robbing never hurts except through the neighbour rule.

> 💡 **Idea.** At each house: rob it (its money + best up to two houses back) or skip it (best up to the previous house). Keep only the last two answers.

**Brute force:** try every non-adjacent subset, O(2ⁿ).

**Steps**
1. `prev` = best up to `i − 2`, `cur` = best up to `i − 1`.
2. For each house: new best = `max(cur, prev + x)`.
3. Shift `prev, cur`.
4. Return `cur`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return which houses to rob?** Keep the full dp array, then walk back from the end: if `dp[i] != dp[i-1]`, house i was robbed — jump to i − 2.
- **Houses in a circle (LC 213)?** Run it twice: without the first house, and without the last. Take the max.
- **Houses in a tree (LC 337)?** Each node returns `(rob, skip)` from post-order DFS.
- **Can't rob within k houses of each other?** `dp[i] = max(dp[i-1], nums[i] + dp[i-k-1])`.

In [4]:
def rob(nums):
    """Max loot from non-adjacent houses."""
    n = len(nums)
    # dp[i] = best loot from the first i houses (dp[0] = 0: no houses, no loot)
    dp = [0] * (n + 1)
    # O(1) space: prev, cur = 0, 0
    for i in range(1, n + 1):
        # skip this house, or rob it on top of the best from two houses back
        skip = dp[i - 1]
        rob_it = nums[i - 1] + (dp[i - 2] if i >= 2 else 0)
        dp[i] = max(skip, rob_it)
        # O(1) space: prev, cur = cur, max(cur, prev + nums[i - 1])
    return dp[n]
    # O(1) space: return cur


assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
print(rob([2, 7, 9, 3, 1]))

12


### 1.1.4 [213. House Robber II](https://leetcode.com/problems/house-robber-ii/) — ★★★ · Medium

**Problem.** Same as House Robber, but houses are in a circle (first and last are adjacent).  
**Example.** `[2, 3, 2]` → `3` (the first and last houses touch, so rob only the 3)  
**Spot it.** Break the circle: the answer excludes either the first or the last house.

**Ask first**
- Only one house? → Return it directly (both slices would be empty).

> 💡 **Idea.** The first and last house touch, so you can't rob both. Run House Robber on `nums[1:]` and on `nums[:-1]`; take the max.

**Brute force:** try all subsets with the circular rule, O(2ⁿ).

**Steps**
1. A single house → its value.
2. Run House Robber on `nums[1:]`.
3. Run House Robber on `nums[:-1]`.
4. Return the larger.

**Cost:** Time O(n) · Space O(n) for the slices (O(1) with index ranges)

**Likely follow-ups**
- **Why are two linear runs enough?** Every valid plan skips the first or the last house (or both), so it's inside one of the two runs.
- **Return the houses?** Do the backtrack on whichever run won.
- **Tree version (LC 337)?** Post-order DFS returning `(rob, skip)`.

In [ ]:
def rob(nums):
    """Max loot from non-adjacent houses."""
    n = len(nums)
    # dp[i] = best loot from the first i houses (dp[0] = 0: no houses, no loot)
    dp = [0] * (n + 1)
    # O(1) space: prev, cur = 0, 0
    for i in range(1, n + 1):
        # skip this house, or rob it on top of the best from two houses back
        skip = dp[i - 1]
        rob_it = nums[i - 1] + (dp[i - 2] if i >= 2 else 0)
        dp[i] = max(skip, rob_it)
        # O(1) space: prev, cur = cur, max(cur, prev + nums[i - 1])
    return dp[n]
    # O(1) space: return cur

def rob_circle(nums):
    """Max loot when houses form a circle."""
    if len(nums) == 1:
        return nums[0]
    
    # first and last can't both be robbed: drop one of them
    return max(rob(nums[1:]), rob(nums[:-1]))


assert rob_circle([2, 3, 2]) == 3
assert rob_circle([1, 2, 3, 1]) == 4
assert rob_circle([1, 2, 3]) == 3
print(rob_circle([2, 3, 2]))

3


### 1.1.5 [740. Delete and Earn](https://leetcode.com/problems/delete-and-earn/) — ★ · Medium

**Problem.** Taking value `x` earns `x` but deletes every `x−1` and `x+1`. Maximise points.  
**Example.** `[3, 4, 2]` → `6` (take the 4 and the 2)  
**Spot it.** Bucket by value → House Robber over the value axis.

**Ask first**
- How big can values get? → ≤ 10⁴ here, so an array works. Huge or sparse values → sort the distinct values instead.

> 💡 **Idea.** Taking value v kills all v − 1 and v + 1. Add up the points per value, then it's House Robber over the values 0..max.

**Brute force:** try every subset of distinct values, O(2ⁿ).

**Steps**
1. `points[v] = v × count(v)`.
2. Run House Robber on `points`.

**Cost:** Time O(n + max value) · Space O(max value)

**Likely follow-ups**
- **Values up to 10⁹ (sparse)?** Sort distinct values; if the next value is exactly +1, it's adjacent (House Robber rule), otherwise you can freely add it. O(n log n).
- **Why does deleting all copies at once make sense?** Once you take one copy of v, its neighbours are gone, so taking the other copies of v costs nothing.

In [ ]:
def rob(nums):
    """Max loot from non-adjacent houses."""
    n = len(nums)
    # dp[i] = best loot from the first i houses (dp[0] = 0: no houses, no loot)
    dp = [0] * (n + 1)
    # O(1) space: prev, cur = 0, 0
    for i in range(1, n + 1):
        # skip this house, or rob it on top of the best from two houses back
        skip = dp[i - 1]
        rob_it = nums[i - 1] + (dp[i - 2] if i >= 2 else 0)
        dp[i] = max(skip, rob_it)
        # O(1) space: prev, cur = cur, max(cur, prev + nums[i - 1])
    return dp[n]
    # O(1) space: return cur

def delete_and_earn(nums):
    """Max points under the delete-neighbours rule."""
    # total earned by taking every copy of each value
    points = [0] * (max(nums) + 1)
    for x in nums:
        points[x] += x
    # neighbouring values behave like neighbouring houses
    return rob(points)


assert delete_and_earn([3, 4, 2]) == 6
assert delete_and_earn([2, 2, 3, 3, 3, 4]) == 9
print(delete_and_earn([2, 2, 3, 3, 3, 4]))

9


### 1.1.6 [91. Decode Ways](https://leetcode.com/problems/decode-ways/) — ★★★ · Medium

**Problem.** Count decodings of a digit string where `A=1 … Z=26`.  
**Example.** `"226"` → `3` (`2 2 6`, `22 6`, `2 26`)  
**Spot it.** Last step consumes one digit (`1–9`) or two digits (`10–26`) → Fibonacci with conditions.

**Ask first**
- Leading or lone `0`? → `0` can't stand alone, so `"0"` or `"30"` give 0 ways.

> 💡 **Idea.** Ways to decode the first i digits = (last digit alone is valid ? ways(i − 1) : 0) + (last two digits form 10–26 ? ways(i − 2) : 0).

**Brute force:** try every split recursively, O(2ⁿ).

**Steps**
1. `prev` = ways for the prefix two shorter, `cur` = ways for the prefix one shorter (0 if `s[0] == "0"`).
2. New count = (`cur` if this digit isn't 0) + (`prev` if the last two digits are 10–26).
3. Shift and repeat.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **O(1) space?** Keep just the last two values.
- **`*` can be any digit 1–9 (LC 639)?** Same DP, but each step multiplies by how many digits/pairs `*` can be (e.g. `*` alone = 9 ways, `1*` = 9, `2*` = 6, `**` = 15), mod 10⁹+7.
- **Return all decodings?** Backtracking — output can be exponential.

In [ ]:
def num_decodings(s):
    """Number of ways to decode s."""
    n = len(s)
    # dp[i] = number of ways to decode the first i characters, s[:i]
    dp = [0] * (n + 1)
    # empty string: one way (do nothing)
    dp[0] = 1
    # first character alone ("0" can't be decoded)
    dp[1] = 1 if s[0] != "0" else 0
    # O(1) space: prev, cur = 1, 1 if s[0] != "0" else 0

    for i in range(2, n + 1):
        # O(1) space: nxt = 0
        # last letter uses one digit: s[i-1] (anything but "0")
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
            # O(1) space: nxt += cur

        # last letter uses two digits: s[i-2:i] (must be 10 to 26)
        if 10 <= int(s[i - 2 : i]) <= 26:
            dp[i] += dp[i - 2]
            # O(1) space: nxt += prev

        # O(1) space: prev, cur = cur, nxt

    return dp[n]
    # O(1) space: return cur


assert num_decodings("12") == 2
assert num_decodings("226") == 3
assert num_decodings("06") == 0
assert num_decodings("2101") == 1
print(num_decodings("226"))

3


### 1.1.7 [790. Domino and Tromino Tiling](https://leetcode.com/problems/domino-and-tromino-tiling/) — ★★ · Medium

**Problem.** Count the ways to tile a `2 × n` board with 2 × 1 dominoes and L-shaped trominoes (any rotation), modulo 10⁹ + 7.  
**Example.** `n = 3` → `5`  
**Spot it.** Count ways to fill a strip from left to right → DP over columns, with a second state for a column that is only half covered.

**Ask first**
- Answer mod 10⁹ + 7? → Yes.

> 💡 **Idea.** Track two states per column: fully covered (`full[i]`) and covered with one square sticking out (`part[i]`). A tromino turns a full board into a part board and back. This simplifies to `dp[n] = 2·dp[n−1] + dp[n−3]`.

**Brute force:** try all placements, exponential.

**Steps**
1. `full[0] = full[1] = 1` and `part[1] = 0`.
2. For `i ≥ 2`: `full[i] = full[i − 1] + full[i − 2] + 2 · part[i − 1]` and `part[i] = part[i − 1] + full[i − 2]`.
3. Return `full[n]` modulo 10⁹ + 7.

**Cost:** Time O(n) (one update per column) · Space O(1) (only the last few values are kept)

**Likely follow-ups**
- **Huge n (like 10¹⁸)?** The recurrence is linear, so use a 3×3 matrix raised to the n-th power by fast exponentiation: O(log n).
- **Only dominoes (no trominoes)?** It's Fibonacci.
- **A 3×n board?** Same idea with more profile states (bitmask of which cells stick out).

In [ ]:
def num_tilings(n):
    """Domino + tromino tilings of a 2 × n board, modulo 1e9 + 7."""
    mod = 10**9 + 7
    # full[i] = tilings of a 2 × i board
    # part[i] = tilings of a 2 × i board with one corner square missing
    full = [0] * (n + 1)
    part = [0] * (n + 1)
    full[0] = full[1] = 1
    
    # O(1) space: full_prev2, full_prev, part_prev = 1, 1, 0
    for i in range(2, n + 1):
        # vertical domino, two flat dominoes, or a tromino (two mirror images)
        full[i] = (full[i - 1] + full[i - 2] + 2 * part[i - 1]) % mod
        # O(1) space: full_cur = (full_prev + full_prev2 + 2 * part_prev) % mod
        
        # the lone corner is covered by a tromino or by a flat domino
        part[i] = (part[i - 1] + full[i - 2]) % mod
        # O(1) space: part_cur = (part_prev + full_prev2) % mod
        # O(1) space: full_prev2, full_prev, part_prev = full_prev, full_cur, part_cur
    return full[n]
    # O(1) space: return full_prev


assert [num_tilings(n) for n in range(1, 6)] == [1, 2, 5, 11, 24]
print(num_tilings(3))

5


### 1.1.8 [983. Minimum Cost For Tickets](https://leetcode.com/problems/minimum-cost-for-tickets/) — ★ · Medium

**Problem.** Travel days in a year; 1-, 7- and 30-day passes with given costs. Minimum total cost?  
**Example.** `days = [1, 4, 6, 7, 8, 20], costs = [2, 7, 15]` → `11`  
**Spot it.** `dp[day]` = cheapest cost to cover all travel days up to `day`.

**Ask first**
- Do passes cover consecutive *calendar* days (not just travel days)? → Yes.

> 💡 **Idea.** `dp[d]` = cheapest way to cover travel up to day d. On a travel day: min of (1-day pass + dp[d−1], 7-day + dp[d−7], 30-day + dp[d−30]). On a non-travel day: `dp[d] = dp[d−1]`.

**Brute force:** try every pass choice per travel day, O(3ⁿ).

**Steps**
1. `dp[0] = 0`.
2. Non-travel day: `dp[d] = dp[d − 1]`.
3. Travel day: `dp[d] = min(cost + dp[max(0, d − span)])` over the three passes.
4. Return `dp[last day]`.

**Cost:** Time O(last day) · Space O(last day)

**Likely follow-ups**
- **Days go up to 10⁹?** DP over the travel-day *indices* instead: for pass length L, jump to the first travel day ≥ `days[i] + L` (two pointers or binary search). O(n).
- **More pass types?** Add one option per pass; O(n · passes).
- **Return which passes to buy?** Store the choice that won at each day and walk back.

In [ ]:
def mincost_tickets(days, costs):
    """Cheapest pass combination covering all days."""
    travel = set(days)

    # dp[d] = cheapest cover of all travel days up to day d
    dp = [0] * (days[-1] + 1)

    for d in range(1, days[-1] + 1):

        # no travel: nothing new to pay for
        if d not in travel:
            dp[d] = dp[d - 1]
            continue

        cost_1_day = dp[max(0, d - 1)] + costs[0]
        cost_7_day = dp[max(0, d - 7)] + costs[1]
        cost_30_day = dp[max(0, d - 30)] + costs[2]

        # keep the cheapest of the three
        dp[d] = min(cost_1_day, cost_7_day, cost_30_day)
    
    return dp[-1]


assert mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11
assert mincost_tickets([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [2, 7, 15]) == 17
print(mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]))

11


## 1.2 Kadane and subarray DP

The state is "best subarray **ending at** `i`". Each element either extends the previous best or starts fresh; the answer is the best over all `i`.

### 1.2.1 [53. Maximum Subarray](https://leetcode.com/problems/maximum-subarray/) — ★★★ · Medium

**Problem.** Largest sum of a non-empty contiguous subarray.  
**Example.** `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` → `6` (`[4, -1, 2, 1]`)  
**Spot it.** Best subarray *ending at i* → Kadane's algorithm.

**Ask first**
- All negative? → Return the largest single number (the subarray can't be empty).

> 💡 **Idea.** Kadane: at each number, either keep adding to the current run or start fresh here — start fresh if the run so far is negative. Track the best ever.

**Brute force:** sum every subarray, O(n²).

**Steps**
1. `cur` = best sum ending here; `best` = best overall.
2. For each `x`: `cur = max(x, cur + x)`.
3. `best = max(best, cur)`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return the start and end indices?** Remember where the current run started; when you update the best, save `(start, i)`.
- **Divide and conquer (the official follow-up)?** Best is in the left half, the right half, or crosses the middle (best suffix of left + best prefix of right). O(n log n).
- **2D: max-sum rectangle?** Fix a top and bottom row, collapse columns into a 1D array, run Kadane. O(rows²·cols).
- **Circular array (LC 918)?** Max of normal Kadane and `total − min subarray`.

In [ ]:
def max_sub_array(nums):
    """Largest contiguous subarray sum."""
    n = len(nums)
    # dp[i] = best subarray sum that ENDS at index i
    dp = [0] * n
    dp[0] = nums[0]
    
    # O(1) space: best = cur = nums[0]
    for i in range(1, n):
        # extend the previous subarray, or start a new one here
        dp[i] = max(nums[i], dp[i - 1] + nums[i])
        # O(1) space: cur = max(nums[i], cur + nums[i])
        # O(1) space: best = max(best, cur)
    return max(dp)
    # O(1) space: return best


assert max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_sub_array([1]) == 1
assert max_sub_array([-3, -1, -2]) == -1
print(max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


### 1.2.2 [152. Maximum Product Subarray](https://leetcode.com/problems/maximum-product-subarray/) — ★★★ · Medium

**Problem.** Largest product of a contiguous subarray.  
**Example.** `[2, 3, -2, 4]` → `6` (`[2, 3]`)  
**Spot it.** Kadane for products — a negative number swaps max and min.

**Ask first**
- Zeros? → Yes — a zero resets both max and min, so the run restarts after it.

> 💡 **Idea.** A negative number flips big and small. Keep both the max and min product ending here; the new max is the biggest of `x`, `x·max`, `x·min`.

**Brute force:** multiply every subarray, O(n²).

**Steps**
1. `hi`, `lo` = max and min product ending here.
2. For each `x`, candidates are `x`, `hi·x`, `lo·x`.
3. New `hi` = max of them, new `lo` = min of them.
4. Track the best `hi`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return the subarray?** Track start indices for both the max and min runs (swap them when x is negative).
- **Another way to see it?** Between zeros, the best is a prefix or a suffix product — scan left-to-right and right-to-left, resetting at zeros.
- **Overflow?** Guaranteed to fit here; in general use big ints or logs of absolute values plus a sign count.

In [ ]:
def max_product(nums):
    """Largest contiguous subarray product."""
    n = len(nums)
    # hi[i] / lo[i] = largest / smallest product of a subarray that ENDS at index i
    # (a negative number can turn the smallest into the largest, so track both)
    hi = [0] * n
    lo = [0] * n
    hi[0] = lo[0] = nums[0]
    
    # O(1) space: best = hi = lo = nums[0]
    for i in range(1, n):
        x = nums[i]

        # start fresh, or extend the largest / smallest product
        candidates = (x, hi[i - 1] * x, lo[i - 1] * x)

        # O(1) space: candidates = (x, hi * x, lo * x)
        hi[i], lo[i] = max(candidates), min(candidates)
        # O(1) space: hi, lo = max(candidates), min(candidates)
        # O(1) space: best = max(best, hi)
    return max(hi)
    # O(1) space: return best


assert max_product([2, 3, -2, 4]) == 6
assert max_product([-2, 0, -1]) == 0
assert max_product([-2, 3, -4]) == 24
print(max_product([2, 3, -2, 4]))

6


### 1.2.3 [918. Maximum Sum Circular Subarray](https://leetcode.com/problems/maximum-sum-circular-subarray/) — ★★ · Medium

**Problem.** Maximum subarray sum where the array is circular.  
**Example.** `[5, -3, 5]` → `10` (wrap around: 5 + 5)  
**Spot it.** A wrapping subarray = total − (minimum subarray in the middle).

**Ask first**
- All numbers negative? → Then `total − min` would mean an empty subarray; return the normal Kadane max instead.

> 💡 **Idea.** The best circular subarray either doesn't wrap (normal Kadane) or wraps — and a wrapping one is the total minus the *smallest* middle subarray. Take the bigger.

**Brute force:** try every start and length, O(n²).

**Steps**
1. Kadane for the maximum subarray.
2. Kadane for the minimum subarray.
3. If all numbers are negative (`best_max < 0`), return `best_max`.
4. Otherwise return `max(best_max, total − best_min)`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Why `total − min`?** Picking a wrapping subarray = leaving out one contiguous middle block; to maximise what's left, leave out the smallest block.
- **Limit on subarray length?** Prefix sums over the doubled array + a monotonic deque of minimum prefix sums within the window.

In [ ]:
def max_subarray_sum_circular(nums):
    """Largest circular subarray sum."""
    n = len(nums)
    # max_end[i] / min_end[i] = largest / smallest subarray sum that ENDS at index i
    max_end = [0] * n
    min_end = [0] * n
    max_end[0] = min_end[0] = nums[0]
    
    for i in range(1, n):
        # Kadane for the maximum and the minimum at once
        max_end[i] = max(nums[i], max_end[i - 1] + nums[i])
        min_end[i] = min(nums[i], min_end[i - 1] + nums[i])
    best_max, best_min = max(max_end), min(min_end)
    
    # O(1) space (keep only the latest values instead of the lists):
    #   cur_max = cur_min = 0
    #   best_max, best_min = nums[0], nums[0]
    #   for x in nums:
    #       cur_max = max(x, cur_max + x)
    #       cur_min = min(x, cur_min + x)
    #       best_max = max(best_max, cur_max)
    #       best_min = min(best_min, cur_min)
    # wrapping answer = everything except the worst middle block
    
    return best_max if best_max < 0 else max(best_max, sum(nums) - best_min)


assert max_subarray_sum_circular([1, -2, 3, -2]) == 3
assert max_subarray_sum_circular([5, -3, 5]) == 10
assert max_subarray_sum_circular([-3, -2, -3]) == -2
print(max_subarray_sum_circular([5, -3, 5]))

10


## 1.3 Word and partition DP

The state is "can / how many ways / best way to build the **prefix** of length `i`". The transition tries every last piece (a word, a square, a part) that ends at `i`.

### 1.3.1 [139. Word Break](https://leetcode.com/problems/word-break/) — ★★★ · Medium

**Problem.** Can `s` be segmented into dictionary words?  
**Example.** `s = "leetcode", words = ["leet", "code"]` → `True`  
**Spot it.** `dp[i]` = prefix of length `i` is segmentable.

**Ask first**
- Can a word be reused? → Yes, any number of times.
- Dictionary size vs word length? → Loop over word lengths (or a trie) instead of all `j < i` when words are short.

> 💡 **Idea.** `ok[i]` = can the first i characters be split into words? `ok[i]` is true if some word ends at i and `ok[start of that word]` is true.

**Brute force:** try every split recursively, O(2ⁿ).

**Steps**
1. `dp[0] = True` (empty prefix).
2. For each `i`, try `j` from `i − longest` to `i − 1`.
3. If `dp[j]` and `s[j:i]` is a word, set `dp[i] = True` and stop.
4. Return `dp[n]`.

**Cost:** Time O(n · L) checks, each O(L) to slice → O(n · L²) · Space O(n)

**Likely follow-ups**
- **Return all sentences (LC 140)?** Memoised DFS returning the list of sentences for each suffix; output can be exponential.
- **Big dictionary?** Put words in a trie and walk it from each `i` — stops as soon as no word continues.
- **Minimum number of words?** `dp[i] = min(dp[j] + 1)` over valid words `s[j:i]`.

In [ ]:
def word_break(s, word_dict):
    """True if s splits into dictionary words."""
    words = set(word_dict)
    longest = max(len(w) for w in words)

    # dp[i]: the first i characters can be segmented
    dp = [True] + [False] * len(s)

    for i in range(1, len(s) + 1):
        
        # the last word s[j:i] is at most `longest` long
        for j in range(max(0, i - longest), i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    
    return dp[-1]


assert word_break("leetcode", ["leet", "code"])
assert word_break("applepenapple", ["apple", "pen"])
assert not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])
print(word_break("leetcode", ["leet", "code"]))

True


### 1.3.2 [279. Perfect Squares](https://leetcode.com/problems/perfect-squares/) — ★★ · Medium

**Problem.** Fewest perfect squares summing to `n`.  
**Example.** `n = 12` → `3` (4 + 4 + 4)  
**Spot it.** Unbounded knapsack / coin change with coins `1, 4, 9, …`.

**Ask first**
- n up to 10⁴? → Yes, so O(n√n) is fine.

> 💡 **Idea.** `dp[x]` = fewest squares adding up to x = 1 + min over squares `s ≤ x` of `dp[x − s]`. (It's Coin Change with square coins.)

**Brute force:** recursion without memo, exponential.

**Steps**
1. `dp[0] = 0`, everything else large.
2. For each `i`, try every square `s ≤ i`: `dp[i] = min(dp[i], dp[i − s] + 1)`.
3. Return `dp[n]`.

**Cost:** Time O(n √n) · Space O(n)

**Likely follow-ups**
- **BFS version?** Nodes are remainders; subtract a square per step; the first time you hit 0 is the answer.
- **Math shortcut?** The answer is always 1–4 (Lagrange). It's 4 iff `n = 4ᵃ(8b + 7)`; 1 if n is a square; 2 if `n − a²` is a square for some a; else 3. O(√n).
- **Return the squares used?** Keep the choice per `dp[x]` and walk back.

In [ ]:
def num_squares(n):
    """Fewest squares summing to n."""
    squares = [k * k for k in range(1, int(n**0.5) + 1)]

    # dp[i] = fewest squares summing to i (n is a safe upper bound)
    dp = [0] + [n] * n

    for i in range(1, n + 1):

        # the last square used is s
        for s in squares:
            if s > i:
                break
            dp[i] = min(dp[i], dp[i - s] + 1)
            
    return dp[n]


assert num_squares(12) == 3
assert num_squares(13) == 2
print(num_squares(12))

3


### 1.3.3 [343. Integer Break](https://leetcode.com/problems/integer-break/) — ★★ · Medium

**Problem.** Split `n ≥ 2` into at least two positive integers maximising their product.  
**Example.** `n = 10` → `36` (3 × 3 × 4)  
**Spot it.** `dp[i] = max over j of max(j, dp[j]) × max(i−j, dp[i−j])`.

**Ask first**
- Must break into at least 2 parts? → Yes, so `n = 2` → 1 and `n = 3` → 2 (not 3).

> 💡 **Idea.** `dp[i]` = best product for i, where each split `j` uses either `j` itself or `dp[j]` (split further) on each side.

**Brute force:** try all partitions, exponential.

**Steps**
1. `dp[1] = 1`.
2. For each `i`, try every cut `j` up to `i / 2`.
3. `dp[i] = max(dp[i], max(j, dp[j]) · max(i − j, dp[i − j]))`.
4. Return `dp[n]`.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **O(1) math answer?** Use as many 3s as possible; if the remainder is 1, swap a 3 + 1 for 2 + 2. So `3^(n//3)`, adjusted for remainders 1 and 2.
- **Huge n with mod 10⁹+7?** Same formula with fast modular exponentiation, O(log n).
- **Why 3s?** For a fixed sum, splitting into parts close to e ≈ 2.7 maximises the product; 3 is the nearest integer.

In [ ]:
def integer_break(n):
    """Max product of a split of n into >= 2 parts."""
    # dp[i] = best product you can get by splitting i into 2 or more parts
    dp = [0] * (n + 1)

    for i in range(2, n + 1):
        # try every cut: the left piece is j, the right piece is what is left
        for j in range(1, i):
            left = j
            right = i - j

            # a piece can stay whole, or be split further for a bigger product
            best_left = max(left, dp[left])
            best_right = max(right, dp[right])

            # keep the biggest product over all the cuts
            dp[i] = max(dp[i], best_left * best_right)

    return dp[n]


assert integer_break(2) == 1
assert integer_break(10) == 36
print(integer_break(10))

36


### 1.3.4 [96. Unique Binary Search Trees](https://leetcode.com/problems/unique-binary-search-trees/) — ★★ · Medium

**Problem.** Number of structurally unique BSTs with keys `1..n`.  
**Example.** `n = 3` → `5`  
**Spot it.** Choose the root, multiply the counts of left and right subtrees → Catalan numbers.

**Ask first**
- Count only, or build the trees? → Count here. Building them is LC 95.

> 💡 **Idea.** Pick each value as the root: left subtree uses the smaller values, right uses the bigger. `count(n) = Σ count(i − 1) · count(n − i)`.

**Brute force:** generate all trees and count them.

**Steps**
1. `g[0] = 1` (empty tree).
2. For each size `nodes`, sum over roots: `g[root − 1] · g[nodes − root]`.
3. Return `g[n]`.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **Generate all the trees (LC 95)?** Recursively build all trees for `[lo, hi]`: for each root, combine every left tree with every right tree. Output is Catalan-sized.
- **Closed form?** The n-th Catalan number: C(2n, n) / (n + 1).

In [ ]:
def num_trees(n):
    """Count of unique BSTs with n keys."""
    # g[k] = number of BSTs with k keys
    g = [1] + [0] * n
    
    for nodes in range(1, n + 1):
        for root in range(1, nodes + 1):
            # left and right subtrees are chosen independently
            g[nodes] += g[root - 1] * g[nodes - root]
    return g[n]


assert num_trees(3) == 5
assert num_trees(1) == 1
print([num_trees(n) for n in range(1, 8)])

[1, 2, 5, 14, 42, 132, 429]


## 1.4 Longest increasing subsequence

`dp[i]` = best subsequence **ending at** `i`, built from any earlier `j` that can come before it. O(n²) is the default; patience sorting + binary search gives O(n log n) for the length.

### 1.4.1 [300. Longest Increasing Subsequence](https://leetcode.com/problems/longest-increasing-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest strictly increasing subsequence.  
**Example.** `[10, 9, 2, 5, 3, 7, 101, 18]` → `4` (for example `2, 3, 7, 101`)  
**Spot it.** Best chain *ending at each position* → `dp[i]` looks back at every earlier smaller number.

**Ask first**
- Strictly increasing? → Yes (use `bisect_left`; non-decreasing would use `bisect_right`).

> 💡 **Idea.** `dp[i]` = longest increasing run that *ends* at `i`. Look back at every earlier smaller number `j` and extend the best one: `dp[i] = 1 + max(dp[j])`.

**Brute force:** try every subsequence, O(2ⁿ).

**Steps**
1. `dp[i] = 1` for every `i` (a number alone is a chain of length 1).
2. For each `i`, look at every earlier `j`; if `nums[j] < nums[i]`, then `dp[i] = max(dp[i], dp[j] + 1)`.
3. The answer is `max(dp)`.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **Return the actual subsequence?** Store, for each number, the index it extended (parent) and the index sitting at each tail; walk back from the last tail.
- **Count the number of LIS (LC 673)?** O(n²) DP with both `length[i]` and `count[i]`; sum counts at the max length.
- **2D version — Russian dolls (LC 354)?** Sort by width ascending and height *descending*, then LIS on heights.
- **O(n log n) (the official follow-up)?** Keep `tails[k]` = smallest possible last number of an increasing run of length k + 1. For each number, binary-search the first tail ≥ it and replace it (or append). `len(tails)` is the answer — `tails` stays sorted, so each step is a binary search.

In [17]:
def length_of_lis(nums):
    """Length of the longest strictly increasing subsequence."""
    n = len(nums)
    # dp[i] = length of the longest increasing subsequence that ENDS at index i
    # (every number alone is a subsequence of length 1)
    dp = [1] * n

    for i in range(n):
        # look at every earlier number j
        for j in range(i):
            # nums[i] can extend a subsequence ending at j if it is bigger
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)

    # the subsequence can end anywhere, so take the best over all ends
    return max(dp)


assert length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert length_of_lis([0, 1, 0, 3, 2, 3]) == 4
assert length_of_lis([7, 7, 7, 7]) == 1
print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))

4


---
# 2) Knapsack

**The gist.** Pick items to fill a capacity (a target sum, an amount, a budget). The table is indexed by capacity: `dp[c]` = best / number of ways / reachable for capacity `c`.

> **Think of it like…** making change at a till: the fewest coins for 11 = one more coin on top of the best of 11 − 1, 11 − 2 and 11 − 5.

| Variant | Loop order | Why |
|---|---|---|
| **0/1** (each item once) | items outer, capacity **downwards** | `dp[c − w]` still holds the value *before* this item |
| **unbounded** (reuse) | capacity **upwards** | `dp[c − w]` may already include this item — allowed |
| count **combinations** | items outer, capacity inner | each combo is built in one fixed item order |
| count **ordered sequences** | capacity outer, items inner | every last item is tried at every capacity |

**Spot it:** "coins", "amount", "target sum", "partition into two equal halves", "assign + / −", "each item once / unlimited".

**Classic bugs:** 0/1 loop going upwards (uses an item twice) · forgetting `dp[0] = 1` when counting · using `inf` math wrong (check `dp[x] != inf` before using it).

## 2.1 Subset sums and coins

Name the capacity, the item weight and what `dp[c]` stores; then pick the loop order from the table above. That choice is the whole difference between these questions.

### 2.1.1 [322. Coin Change](https://leetcode.com/problems/coin-change/) — ★★★ · Medium

**Problem.** Fewest coins summing to `amount` (unlimited coins), or −1.  
**Example.** `coins = [1, 2, 5], amount = 11` → `3` (5 + 5 + 1)  
**Spot it.** Unbounded knapsack minimising count.

**Ask first**
- Unlimited coins of each kind? → Yes (unbounded knapsack).
- Impossible amount? → Return −1.

> 💡 **Idea.** `dp[a]` = fewest coins to make amount a = 1 + min over coins of `dp[a − coin]`. Unreachable stays `inf`.

**Brute force:** greedy (biggest coin first) — wrong, e.g. coins `[1, 3, 4]`, amount 6: greedy gives 3 coins, best is 2.

**Steps**
1. `dp[0] = 0`, everything else "infinity" (`amount + 1`).
2. For each amount `a`, for each coin `c ≤ a`: `dp[a] = min(dp[a], dp[a − c] + 1)`.
3. Return `dp[amount]`, or −1 if it is still infinity.

**Cost:** Time O(amount · coins) · Space O(amount)

**Likely follow-ups**
- **Return which coins were used?** Store the last coin used for each amount; walk back from the target.
- **BFS version?** Amounts are nodes, each coin is an edge; the BFS level when you reach the target is the answer.
- **When is greedy safe?** Only for "canonical" coin systems like 1, 5, 10, 25 — prove it or use DP.
- **Limited count per coin?** Bounded knapsack: loop each coin's copies, or use binary splitting.

In [ ]:
def coin_change(coins, amount):
    """Fewest coins for amount, or -1."""
    # amount + 1 coins is impossible, so it acts as infinity
    inf = amount + 1
    dp = [0] + [inf] * amount
    
    for a in range(1, amount + 1):
        # try each coin as the last one used
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] < inf else -1


assert coin_change([1, 2, 5], 11) == 3
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
assert coin_change([1, 3, 4], 6) == 2
print(coin_change([1, 2, 5], 11))

3


### 2.1.2 [518. Coin Change II](https://leetcode.com/problems/coin-change-ii/) — ★★ · Medium

**Problem.** Number of *combinations* of coins that make `amount`.  
**Example.** `amount = 5, coins = [1, 2, 5]` → `4`  
**Spot it.** Unbounded knapsack counting; coins in the outer loop so each combination counts once.

**Ask first**
- Do `1+2` and `2+1` count as the same way? → Yes (combinations) — that decides the loop order.

> 💡 **Idea.** Count combinations: coins in the **outer** loop, amounts in the inner loop, `dp[a] += dp[a − coin]`. Fixed coin order means each combination is counted once.

**Brute force:** backtracking over coins, exponential.

**Steps**
1. `dp[0] = 1` (one way to make 0: no coins).
2. For each coin, for `a` from `coin` up to `amount`: `dp[a] += dp[a − coin]`.
3. Return `dp[amount]`.

**Cost:** Time O(amount · coins) · Space O(amount)

**Likely follow-ups**
- **Swap the loops — what happens?** Amount outer, coins inner counts *ordered* sequences (that's LC 377).
- **Each coin at most once?** 0/1 knapsack: loop amounts *downwards*.
- **Answer can be huge?** Take it mod a prime if asked; Python ints are fine otherwise.

In [ ]:
def change(amount, coins):
    """Number of coin combinations for amount."""
    dp = [1] + [0] * amount
    
    # coins outer: each combination counted in one coin order
    for c in coins:
        # upwards: the same coin may be used again
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]


assert change(5, [1, 2, 5]) == 4
assert change(3, [2]) == 0
assert change(10, [10]) == 1
print(change(5, [1, 2, 5]))

4


### 2.1.3 [377. Combination Sum IV](https://leetcode.com/problems/combination-sum-iv/) — ★★★ · Medium

**Problem.** Number of *ordered* sequences of `nums` (reuse allowed) summing to `target`.  
**Example.** `nums = [1, 2, 3], target = 4` → `7`  
**Spot it.** Counting sequences → target in the outer loop.

**Ask first**
- Do `1+2` and `2+1` count separately? → Yes — that's why it's amounts-outer.

> 💡 **Idea.** Order matters here, so loop the **target** outside and the numbers inside: `dp[t] += dp[t − x]` — every possible last number is tried at every total.

**Brute force:** backtracking over all ordered sequences, exponential.

**Steps**
1. `dp[0] = 1`.
2. For each `t` from 1 to target: `dp[t] = Σ dp[t − x]` over `x ≤ t`.
3. Return `dp[target]`.

**Cost:** Time O(target · n) · Space O(target)

**Likely follow-ups**
- **Negative numbers allowed?** Infinitely many sequences (`+1, −1` repeated). You'd need a cap on sequence length, then DP over `(length, sum)`.
- **Return the sequences?** Backtracking — output can be huge.
- **How is it different from Coin Change II?** Same recurrence, loops swapped: combinations vs ordered sequences.

In [ ]:
def combination_sum4(nums, target):
    """Ordered ways to sum to target."""
    dp = [1] + [0] * target
    
    # target outer: every number can be the last element at every total
    for t in range(1, target + 1):
        # try every number x as the LAST one in the sequence
        for x in nums:
            # x only fits if it is not bigger than the total t
            if x <= t:
                # the rest (t - x) can be made in dp[t - x] ways
                dp[t] += dp[t - x]
    return dp[target]


assert combination_sum4([1, 2, 3], 4) == 7
assert combination_sum4([9], 3) == 0
print(combination_sum4([1, 2, 3], 4))

7


### 2.1.4 [416. Partition Equal Subset Sum](https://leetcode.com/problems/partition-equal-subset-sum/) — ★★ · Medium

**Problem.** Can the array be split into two subsets with equal sums?  
**Example.** `[1, 5, 11, 5]` → `True` (`[1, 5, 5]` and `[11]`)  
**Spot it.** Subset sum to `total / 2` → 0/1 knapsack on booleans.

**Ask first**
- Is the total odd? → Then it's impossible; return False right away.
- All positive? → Yes, so the target is bounded by total / 2.

> 💡 **Idea.** Two equal halves ⇔ some subset sums to `total / 2`. 0/1 knapsack reachability: for each number, go *downwards* `can[s] |= can[s − x]`.

**Brute force:** try every subset, O(2ⁿ).

**Steps**
1. Odd total → `False`.
2. `dp[0] = True`; `dp[s]` = some subset sums to `s`.
3. For each number `x`, for `s` from `half` down to `x`: `dp[s] |= dp[s − x]`.
4. Return `dp[half]`.

**Cost:** Time O(n · total) · Space O(total)

**Likely follow-ups**
- **Faster in Python?** Bitset: `bits |= bits << x`, then check bit `total // 2`.
- **Return the two subsets?** Keep a 2D table (or the item that first reached each sum) and walk back from the target.
- **k equal subsets (LC 698)?** Backtracking / bitmask DP — the knapsack trick doesn't extend.
- **Negative numbers allowed?** Shift sums by the total of negatives, or use a set of reachable sums.

In [ ]:
def can_partition(nums):
    """True if nums splits into two equal-sum halves."""
    total = sum(nums)

    if total % 2:
        return False
    half = total // 2

    # dp[s]: some subset of the numbers seen so far sums to s
    dp = [True] + [False] * half

    for x in nums:
        # downwards so x is used at most once
        for s in range(half, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
            
    return dp[half]


assert can_partition([1, 5, 11, 5])
assert not can_partition([1, 2, 3, 5])
print(can_partition([1, 5, 11, 5]))

True


### 2.1.5 [494. Target Sum](https://leetcode.com/problems/target-sum/) — ★★ · Medium

**Problem.** Assign `+` or `−` to each number; count assignments evaluating to `target`.  
**Example.** `nums = [1, 1, 1, 1, 1], target = 3` → `5`  
**Spot it.** Every number branches into `+x` or `−x` → keep a list of how many ways reach each running total.

**Ask first**
- Can numbers be 0? → Yes — each zero can be + or −, so it doubles the count (the table handles it: `t + 0` and `t − 0` both get added).
- `total + target` odd, or `|target| > total`? → Return 0.

> 💡 **Idea.** Each number either adds or subtracts. Keep a table of how many ways reach each running total; every number spreads each total `t` to `t + x` and `t − x`.

**Brute force:** try all 2ⁿ sign combinations.

**Steps**
1. If `|target| > total`, return 0. Running totals range from `−total` to `+total`, so shift by `total`: running total `t` lives at `dp[t + total]`.
2. `dp[total] = 1`: before any number, total 0 is reached in one way.
3. For each number `x`, build `new_dp`: every reachable total `t` adds its count to `t + x` and to `t − x`.
4. Replace `dp` with `new_dp`. Return `dp[target + total]`.

**Cost:** Time O(n · total) · Space O(total) (for the list of shifted totals)

**Likely follow-ups**
- **Smaller table (subset-sum trick)?** Split into a `+` group P and a `−` group N: `P − N = target` and `P + N = total` give `P = (total + target) / 2`. Count subsets summing to P with a 0/1 knapsack (loop sums downwards).
- **Print the actual expressions?** Backtracking over signs (or walk back through a 2D table) — output can be 2ⁿ.
- **Numbers up to 10⁹ but n ≤ 40?** Meet in the middle: count sums of each half in a dict, then match.

In [ ]:
def find_target_sum_ways(nums, target):
    """Number of +/- assignments reaching target."""

    total = sum(nums)

    # the target is out of reach even if every sign is the same
    if abs(target) > total:
        return 0

    dp = [0] * (2 * total + 1)
    
    # before any number: total 0, in one way
    dp[0 + total] = 1

    for x in nums:
        new_dp = [0] * (2 * total + 1)
        for t in range(-total, total + 1):
            ways = dp[t + total]
            # skip totals we cannot reach
            if ways == 0:
                continue
            # put + in front of x
            new_dp[t + x + total] += ways
            # put - in front of x
            new_dp[t - x + total] += ways
        dp = new_dp

    return dp[target + total]


assert find_target_sum_ways([1, 1, 1, 1, 1], 3) == 5
assert find_target_sum_ways([1], 1) == 1
assert find_target_sum_ways([0, 0, 1], 1) == 4
print(find_target_sum_ways([1, 1, 1, 1, 1], 3))

5


### 2.1.6 [1049. Last Stone Weight II](https://leetcode.com/problems/last-stone-weight-ii/) — ★★ · Medium

**Problem.** Smash stones pairwise (result `|a − b|`); minimise the last remaining weight.  
**Example.** `[2, 7, 4, 1, 8, 1]` → `1`  
**Spot it.** The result is `|sum(A) − sum(B)|` for some split → subset sum closest to `total / 2`.

**Ask first**
- Is the order of smashes free? → Yes — any sign split can be reached.

> 💡 **Idea.** Smashing stones is like giving each a + or − sign. Split into two piles as equal as possible: find the biggest subset sum ≤ total / 2; answer = total − 2 × that.

**Brute force:** simulate every smash order, factorial.

**Steps**
1. Build the set of reachable subset sums up to `total // 2`.
2. Take the largest reachable `s`.
3. Answer = `total − 2s`.

**Cost:** Time O(n · total) · Space O(total)

**Likely follow-ups**
- **Why isn't the greedy heap (LC 1046) right here?** 1046 forces smashing the two heaviest; here you choose, so it's a partition problem.
- **Bitset trick?** `bits |= bits << w`, then take the largest set bit ≤ total // 2.
- **Return which stones go in each pile?** Walk back through a 2D reachability table.

In [ ]:
def last_stone_weight_ii(stones):
    """Smallest possible final stone weight."""
    total = sum(stones)
    
    # split the stones into two piles; 
    half = total // 2

    # reach = every pile weight we can build so far (the empty pile weighs 0)
    reach = {0}

    for x in stones:
        # new weights made by adding this stone to each pile we already have
        new_weights = set()
        for weight in reach:
            # a pile heavier than half is never the smaller pile, so ignore it
            if weight + x <= half:
                new_weights.add(weight + x)
        # keep the old piles too (we may skip this stone)
        reach = reach | new_weights

    # the heaviest small pile leaves the smallest gap between the two piles
    biggest_small_pile = max(reach)
    return total - 2 * biggest_small_pile


assert last_stone_weight_ii([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight_ii([31, 26, 33, 21, 40]) == 5
print(last_stone_weight_ii([2, 7, 4, 1, 8, 1]))

1


---
# 3) State Machine DP

**The gist.** When what you can do today depends on a *mode* (holding a share, cooling down, trades used), keep one best value per mode and update all modes each day.

> **Think of it like…** a traffic light: the next colour depends only on the current one. Draw the modes as boxes and the allowed changes as arrows — each arrow is one line of code.

**Spot it:** stock buy/sell with limits, cooldown, fee · "can't do X twice in a row".

**Classic bugs:** updating one mode with *today's* value of another mode when it should use yesterday's (use simultaneous assignment or temporaries).

## 3.1 Stock problems

Modes: `hold` (own a share; best balance so far) and `cash` / `sold` / `rest` (own nothing). Each day: new hold = max(keep holding, buy from a free mode); new cash = max(keep cash, sell from hold).

### 3.1.1 [122. Best Time to Buy and Sell Stock II](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-ii/) — ★★ · Medium

**Problem.** Unlimited transactions (hold at most one share); maximise profit.  
**Example.** `[7, 1, 5, 3, 6, 4]` → `7` (buy 1 sell 5, buy 3 sell 6)  
**Spot it.** Two states (holding / not holding) — or greedily take every upward move.

**Ask first**
- Can you buy and sell on the same day? → Yes, which is why summing rises works.

> 💡 **Idea.** Unlimited trades: take every price rise. Sum `max(0, p[i] − p[i−1])`.

**Brute force:** try every set of buy/sell days, exponential.

**Steps**
1. For each pair of consecutive days, compute the change.
2. Add it if positive.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **At most k trades (LC 188)?** DP with `buy[j]`, `sell[j]` for j = 1..k: O(n·k). If `k ≥ n / 2`, it's unlimited — use this greedy.
- **At most 2 trades (LC 123)?** Four variables: `buy1, sell1, buy2, sell2`.
- **Return the trades?** Each maximal rising run is one buy (start) and one sell (end).

In [24]:
def max_profit_ii(prices):
    """Profit with unlimited transactions."""
    profit = 0

    # look at each pair of neighbouring days: yesterday and today
    for i in range(1, len(prices)):
        yesterday = prices[i - 1]
        today = prices[i]

        # every rise can be captured by buying yesterday and selling today
        if today > yesterday:
            profit += today - yesterday

    return profit


assert max_profit_ii([7, 1, 5, 3, 6, 4]) == 7
assert max_profit_ii([1, 2, 3, 4, 5]) == 4
assert max_profit_ii([7, 6, 4, 3, 1]) == 0
print(max_profit_ii([7, 1, 5, 3, 6, 4]))

7


### 3.1.2 [309. Best Time to Buy and Sell Stock with Cooldown](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-cooldown/) — ★★ · Medium

**Problem.** Unlimited transactions, but after selling you must wait one day before buying.  
**Example.** `[1, 2, 3, 0, 2]` → `3`  
**Spot it.** Three states: `hold`, `sold` (just sold today), `rest` (free to buy).

**Ask first**
- Cooldown only after selling? → Yes, one day.

> 💡 **Idea.** Three modes: `hold` (own a share), `sold` (just sold today), `rest` (free to buy). Buying only from `rest` enforces the 1-day cooldown.

**Brute force:** try every action per day, O(3ⁿ).

**Steps**
1. `hold = −∞`, `sold = rest = 0`.
2. Each day, simultaneously: `hold = max(hold, rest − p)`, `sold = hold + p`, `rest = max(rest, sold)`.
3. Return `max(sold, rest)`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Cooldown of k days?** Buy from the cash state of `i − k − 1`: `hold[i] = max(hold[i−1], cash[i−k−1] − p[i])` — keep a small history.
- **Plus a fee?** Subtract the fee when selling.
- **Return the trades?** Store the choice per day and walk back.

In [ ]:
def max_profit_cooldown(prices):
    """Profit with a one-day cooldown after selling."""

    hold = float("-inf")
    sold = 0
    rest = 0
    # list version: hold, sold, rest = [float("-inf")] * (n + 1), [0] * (n + 1), [0] * (n + 1)

    for p in prices:
        # build today's values from yesterday's values
        new_hold = max(hold, rest - p)
        new_sold = hold + p
        new_rest = max(rest, sold)
        # list version: hold[i], sold[i], rest[i] = new_hold, new_sold, new_rest

        # today becomes yesterday for the next round
        hold, sold, rest = new_hold, new_sold, new_rest

    return max(sold, rest)
    # list version: return max(sold[n], rest[n])


assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert max_profit_cooldown([1]) == 0
print(max_profit_cooldown([1, 2, 3, 0, 2]))

3


### 3.1.3 [714. Best Time to Buy and Sell Stock with Transaction Fee](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-transaction-fee/) — ★ · Medium

**Problem.** Unlimited transactions, each sale costs `fee`.  
**Example.** `prices = [1, 3, 2, 8, 4, 9], fee = 2` → `8`  
**Spot it.** Two states with the fee charged on selling.

**Ask first**
- Fee per full trade (buy + sell)? → Yes — charge it once, on the sell (or the buy, but not both).

> 💡 **Idea.** Two modes: `cash` and `hold`. Each day `cash = max(cash, hold + price − fee)` and `hold = max(hold, cash − price)`.

**Brute force:** try every buy/sell day, exponential.

**Steps**
1. `cash = 0`, `hold = −prices[0]`.
2. Each day: `cash = max(cash, hold + p − fee)`.
3. `hold = max(hold, cash − p)`.
4. Return `cash`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return the trades?** Keep the day indices that set `hold` and `cash`; walk back.
- **Greedy version?** Track the effective buy price; sell when `price > buy + fee`, then set `buy = price − fee` so a later higher price can extend the trade.
- **Fee *and* cooldown?** Combine: three modes with the fee on the sell transition.

In [ ]:
def max_profit_fee(prices, fee):
    """Profit with a per-transaction fee."""

    n = len(prices)
    cash = 0
    hold = -prices[0]

    # O(1) space: cash, hold = 0, -prices[0]
    for i in range(1, n):
        p = prices[i]

        # sell today (paying the fee) or stay in cash
        cash = max(cash, hold + p - fee)

        # buy today or keep holding
        hold = max(hold, cash - p)

    return cash

assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert max_profit_fee([1, 3, 7, 5, 10, 3], 3) == 6
print(max_profit_fee([1, 3, 2, 8, 4, 9], 2))

8


---
# 4) Grid DP

**The gist.** When you can only move right/down, a cell's answer comes from the cells you could have arrived from. Fill the grid in reading order and each cell reads finished neighbours.

> **Think of it like…** counting routes through a grid of one-way streets: routes to a crossing = routes to the crossing above + routes to the one on the left.

`dp[r][c]` only needs the row above and the cell to the left, so **one row array** updated in place is enough.

**Spot it:** "unique paths", "minimum path sum", moves only right/down, "largest square of 1s".

**Classic bugs:** first row/column base cases (an obstacle blocks everything after it) · trying grid DP when moves can go in all 4 directions (that's BFS/Dijkstra).

## 4.1 Paths on grids

For each cell, list where you could have come from; combine those cells (`+` for counting, `min` for cost). Then pick the fill order and squeeze it to one row.

### 4.1.1 [62. Unique Paths](https://leetcode.com/problems/unique-paths/) — ★★★ · Medium

**Problem.** Robot moves only right or down in an `m × n` grid; number of paths from top-left to bottom-right.  
**Example.** `m = 3, n = 2` → `3`  
**Spot it.** `dp[r][c] = dp[r−1][c] + dp[r][c−1]`.

**Ask first**
- Only right and down? → Yes.

> 💡 **Idea.** Routes to a cell = routes from above + routes from the left. One row array: `row[c] += row[c − 1]`.

**Brute force:** recursion trying both moves, O(2^(m+n)).

**Steps**
1. The first row is all 1s.
2. For every next row, `row[c] += row[c − 1]` (above + left).
3. Return `row[-1]`.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **O(1) math answer?** You make `m − 1` downs and `n − 1` rights in some order: C(m + n − 2, m − 1).
- **Huge grid, answer mod 10⁹+7?** Factorials with modular inverses (Fermat), O(m + n).
- **Obstacles (LC 63)?** Set blocked cells to 0 in the DP.

In [ ]:
def unique_paths(m, n):
    """Lattice paths in an m x n grid."""
    # top row: only one way to each cell (straight line)
    row = [1] * n

    for _ in range(1, m):
        for c in range(1, n):
            row[c] += row[c - 1]

    return row[-1]

assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
print(unique_paths(3, 7))

28


### 4.1.2 [63. Unique Paths II](https://leetcode.com/problems/unique-paths-ii/) — ★★ · Medium

**Problem.** Same with obstacles (`1` cells cannot be entered).  
**Example.** `[[0,0,0],[0,1,0],[0,0,0]]` → `2`  
**Spot it.** Unique Paths with `dp = 0` on obstacles.

**Ask first**
- Can the start or end be blocked? → Yes → answer 0.

> 💡 **Idea.** Same as Unique Paths, but an obstacle cell has 0 routes.

**Brute force:** recursion, exponential.

**Steps**
1. `row[0] = 1`, the rest 0.
2. For each cell: obstacle → 0; otherwise add the left value (column 0 keeps the value from above).
3. Return `row[-1]`.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **O(n) space?** One row array: `row[c] = 0 if blocked else row[c] + row[c − 1]`.
- **Huge grid with a few obstacles?** Count paths with C(·) between sorted obstacles and subtract paths through earlier obstacles (inclusion–exclusion), O(k²).
- **Must collect some cells / pass a checkpoint?** Paths(start→checkpoint) × paths(checkpoint→end).

In [ ]:
def unique_paths_with_obstacles(grid):
    """Lattice paths avoiding obstacles."""

    rows, cols = len(grid), len(grid[0])

    dp = [[0] * cols for _ in range(rows)]

    for r in range(rows):
        for c in range(cols):

            # no path enters an obstacle
            if grid[r][c]:
                dp[r][c] = 0
            # the start has one path
            elif r == 0 and c == 0:
                dp[r][c] = 1
            # otherwise: from above plus from the left
            else:
                from_above = dp[r - 1][c] if r > 0 else 0
                from_left = dp[r][c - 1] if c > 0 else 0
                dp[r][c] = from_above + from_left

    return dp[rows - 1][cols - 1]
    # 1-row version: return row[-1]


assert unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert unique_paths_with_obstacles([[0, 1], [0, 0]]) == 1
assert unique_paths_with_obstacles([[1]]) == 0
print(unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))

2


### 4.1.3 [64. Minimum Path Sum](https://leetcode.com/problems/minimum-path-sum/) — ★★ · Medium

**Problem.** Minimum sum path from top-left to bottom-right moving right/down.  
**Example.** `[[1,3,1],[1,5,1],[4,2,1]]` → `7` (1 → 3 → 1 → 1 → 1)  
**Spot it.** `dp[r][c] = grid[r][c] + min(up, left)`.

**Ask first**
- Only right and down? → Yes. In all 4 directions it's Dijkstra.

> 💡 **Idea.** Cheapest way into a cell = its cost + min(cell above, cell to the left).

**Brute force:** try every path, exponential.

**Steps**
1. `row` starts as ∞ except `row[0] = 0`.
2. For each cell: `row[c] = cell + min(row[c] (above), row[c − 1] (left))`.
3. Return `row[-1]`.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **Print the path?** Keep the full table; walk back from the end choosing the smaller of up/left.
- **Moves in all 4 directions?** Loops become possible → Dijkstra over cells, O(m·n log(m·n)).
- **O(1) extra space?** Overwrite the grid in place (if allowed).

In [ ]:
def min_path_sum(grid):
    """Cheapest monotone path sum."""
    inf = float("inf")
    rows, cols = len(grid), len(grid[0])

    dp = [[0] * cols for _ in range(rows)]
    # 1-row version: row = [inf] * len(grid[0]); row[0] = 0

    for r in range(rows):
        for c in range(cols):
            # the start cell costs just itself
            if r == 0 and c == 0:
                dp[r][c] = grid[r][c]
                continue
            
            # cells outside the grid count as infinity
            above = dp[r - 1][c] if r > 0 else inf
            left = dp[r][c - 1] if c > 0 else inf
            dp[r][c] = grid[r][c] + min(above, left)

            # 1-row version: left = row[c - 1] if c else inf
            # 1-row version: row[c] = grid[r][c] + min(row[c], left)
            # (row[c] is the cell above)   
    
    return dp[rows - 1][cols - 1]
    # 1-row version: return row[-1]


assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

7


### 4.1.4 [120. Triangle](https://leetcode.com/problems/triangle/) — ★★ · Medium

**Problem.** Minimum path sum from top to bottom of a triangle (move to adjacent numbers below).  
**Example.** `[[2],[3,4],[6,5,7],[4,1,8,3]]` → `11` (2 + 3 + 5 + 1)  
**Spot it.** Bottom-up: each cell picks the better of its two children.

**Ask first**
- Can I modify the input? → If not, copy the last row into an array (O(n) space).

> 💡 **Idea.** Go bottom-up: each cell becomes its value + the smaller of the two cells below it. The top cell ends up holding the answer.

**Brute force:** try every top-to-bottom path, O(2ⁿ).

**Steps**
1. Start with a copy of the last row.
2. For each row upward: `dp[c] = tri[r][c] + min(dp[c], dp[c + 1])`.
3. Return `dp[0]`.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **O(n) extra space (the official follow-up)?** One array the size of the last row, updated going up.
- **Why bottom-up?** The top has one answer, so there's nothing to take a min over at the end.
- **Return the path?** Store which child was chosen at each cell; walk down from the top.

In [ ]:
def minimum_total(triangle):
    """Min top-to-bottom path sum."""
    n = len(triangle)
    # dp[r][c] = best path sum from cell (r, c) down to the bottom row
    dp = [row.copy() for row in triangle]
    
    for r in range(n - 2, -1, -1):
        for c in range(r + 1):
            # step to the better of the two cells below
            dp[r][c] = triangle[r][c] + min(dp[r + 1][c], dp[r + 1][c + 1])

    return dp[0][0]
    # 1-row version: return dp[0]


assert minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
assert minimum_total([[-10]]) == -10
print(minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]))

11


### 4.1.5 [221. Maximal Square](https://leetcode.com/problems/maximal-square/) — ★★ · Medium

**Problem.** Area of the largest square of `"1"`s.  
**Example.** `[["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]` → `4` (a 2×2 square)  
**Spot it.** `dp[r][c]` = side of the largest square with bottom-right corner at `(r, c)`.

**Ask first**
- Return area or side? → Area.

> 💡 **Idea.** `dp[r][c]` = side of the biggest all-1 square ending at (r, c) = 1 + min(up, left, up-left). The answer is the largest side, squared.

**Brute force:** for each cell try every square size, O((m·n)²).

**Steps**
1. One row array with a leading 0, plus a `diag` variable.
2. For each `"1"` cell: `dp[c] = 1 + min(above, left, diag)`; for `"0"`: 0.
3. Save the old `dp[c]` as the next `diag`.
4. Track the largest side; return its square.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **Count all squares of 1s (LC 1277)?** Same DP, sum all `dp` values.
- **Largest rectangle of 1s (LC 85)?** Build a histogram per row and run Largest Rectangle in Histogram (monotonic stack).
- **O(n) space?** One row plus a saved diagonal value.

In [ ]:
def maximal_square(matrix):
    """Area of the largest all-ones square."""
    rows, cols = len(matrix), len(matrix[0])

    dp = [[0] * (cols + 1) for _ in range(rows + 1)]

    best = 0
    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            if matrix[r - 1][c - 1] == "1":
                # limited by the smallest of the three neighbouring squares
                dp[r][c] = 1 + min(dp[r - 1][c], dp[r][c - 1], dp[r - 1][c - 1])
            best = max(best, dp[r][c])

    return best * best


m = [list("10100"), list("10111"), list("11111"), list("10010")]
assert maximal_square(m) == 4
assert maximal_square([list("01"), list("10")]) == 1
print(maximal_square(m))

4


### 4.1.6 [329. Longest Increasing Path in a Matrix](https://leetcode.com/problems/longest-increasing-path-in-a-matrix/) — ★★ · Hard

**Problem.** Length of the longest strictly increasing path (4 directions) in a matrix.  
**Example.** `[[9,9,4],[6,6,8],[2,1,1]]` → `4` (1 → 2 → 6 → 9)  
**Spot it.** DFS + memo: the increasing constraint makes the graph acyclic.

**Ask first**
- Diagonals? → No, 4 directions.

> 💡 **Idea.** Memoised DFS from every cell: longest path from here = 1 + max over strictly bigger neighbours. Values only go up, so there are no cycles.

**Brute force:** DFS from every cell without memo, exponential.

**Steps**
1. `longest(r, c)` = 1 + max of `longest` over larger neighbours.
2. Cache it with `@cache`.
3. Return the max over all cells.

**Cost:** Time O(mn) (each cell computed once, 4 neighbours each) · Space O(mn) (the cache and recursion)

**Likely follow-ups**
- **Without recursion?** Topological sort by value: peel cells with no smaller neighbours layer by layer; the number of layers is the answer.

In [32]:
def longest_increasing_path(matrix):
    """Longest strictly increasing path length."""
    rows, cols = len(matrix), len(matrix[0])

    @cache
    def longest(r, c):
        """Longest increasing path starting at (r, c)."""
        best = 1
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            # only climbing moves: no cycles, so no visited set
            if 0 <= nr < rows and 0 <= nc < cols and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + longest(nr, nc))
        return best

    return max(longest(r, c) for r in range(rows) for c in range(cols))


assert longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]) == 4
assert longest_increasing_path([[3, 4, 5], [3, 2, 6], [2, 2, 1]]) == 4
print(longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]))

4


---
# 5) String DP

**The gist.** Compare prefixes (two strings) or substrings (one string). For two strings, `dp[i][j]` answers the question for `a[:i]` and `b[:j]`, built from what happens to the *last* characters.

> **Think of it like…** comparing two drafts of an essay: if the last lines match, they cost nothing; if not, you delete, insert or edit a line — and the rest is the same comparison on shorter drafts.

**Two strings:** last characters match → extend `dp[i−1][j−1]`; otherwise combine `dp[i−1][j]` (drop from `a`) and `dp[i][j−1]` (drop from `b`). Row 0 / column 0 are the empty-prefix base cases. **One string, palindromes:** expand around centres, or `dp[l][r]` filled by increasing length.

**Spot it:** "longest common…", "edit / convert one string to another", "interleave", "palindromic substring / subsequence".

**Classic bugs:** substring (contiguous) vs subsequence (gaps allowed) · forgetting the empty-string row/column · in 1-row versions, overwriting the diagonal before you read it.

## 5.1 Palindromes

Every palindrome has a centre: one character (odd length) or a gap (even). Expanding from all 2n − 1 centres visits every palindromic substring in O(n²) time, O(1) space.

### 5.1.1 [5. Longest Palindromic Substring](https://leetcode.com/problems/longest-palindromic-substring/) — ★★★ · Medium

**Problem.** Longest palindromic substring.  
**Example.** `"babad"` → `"bab"` (`"aba"` is also correct)  
**Spot it.** Every palindrome has a centre → expand around 2n − 1 centres.

**Ask first**
- Return the substring (not its length)? → Yes — track `start` and `length`.

> 💡 **Idea.** Every palindrome has a centre. Expand from each of the 2n − 1 centres while the ends match; keep the longest.

**Brute force:** check every substring, O(n³).

**Steps**
1. Loop over 2n − 1 centres; centre `k` starts at `lo = k // 2`, `hi = (k + 1) // 2`.
2. Expand while both ends match.
3. The palindrome is `s[lo + 1 : hi]`; keep the longest.

**Cost:** Time O(n²) · Space O(1)

**Likely follow-ups**
- **O(n)?** Manacher's algorithm — reuses mirror information around the current rightmost palindrome. Mention it; rarely coded.
- **DP version?** `pal[i][j] = s[i] == s[j] and pal[i+1][j−1]`, O(n²) time and space.
- **Longest palindromic *subsequence*?** That's LC 516 (interval DP).

In [33]:
def longest_palindrome_substring(s):
    """Longest palindromic substring."""
    best = (0, 0)
    # even k: centre on a character; odd k: centre between two
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        # grow while the two ends match
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        # the last successful palindrome is s[lo + 1 : hi]
        if hi - lo - 1 > best[1] - best[0]:
            best = (lo + 1, hi)
    return s[best[0] : best[1]]


assert longest_palindrome_substring("babad") in ("bab", "aba")
assert longest_palindrome_substring("cbbd") == "bb"
assert longest_palindrome_substring("a") == "a"
print(longest_palindrome_substring("forgeeksskeegfor"))

geeksskeeg


### 5.1.2 [647. Palindromic Substrings](https://leetcode.com/problems/palindromic-substrings/) — ★★★ · Medium

**Problem.** Count palindromic substrings.  
**Example.** `"aaa"` → `6` (`a`, `a`, `a`, `aa`, `aa`, `aaa`)  
**Spot it.** Expand around centres, counting every successful expansion.

**Ask first**
- Same substring at different positions counts twice? → Yes, positions matter.

> 💡 **Idea.** Expand around all 2n − 1 centres; every successful expansion is one more palindrome.

**Brute force:** check every substring, O(n³).

**Steps**
1. Loop over 2n − 1 centres.
2. While the ends match, count one palindrome and expand.
3. Return the count.

**Cost:** Time O(n²) · Space O(1)

**Likely follow-ups**
- **O(n)?** Manacher: sum `(radius + 1) // 2` over the transformed string.
- **Count *distinct* palindromes?** Add each found palindrome to a set (O(n²) memory) — or a palindromic tree (eertree) in O(n).
- **Minimum cuts to split into palindromes (LC 132)?** DP on top of the palindrome table.

In [34]:
def count_substrings(s):
    """Number of palindromic substrings."""
    count = 0
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        # each successful step is one more palindrome
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            count += 1
            lo, hi = lo - 1, hi + 1
    return count


assert count_substrings("abc") == 3
assert count_substrings("aaa") == 6
print(count_substrings("aaa"))

6


### 5.1.3 [516. Longest Palindromic Subsequence](https://leetcode.com/problems/longest-palindromic-subsequence/) — ★ · Medium

**Problem.** Length of the longest palindromic subsequence.  
**Example.** `"bbbab"` → `4` (`"bbbb"`)  
**Spot it.** Interval DP `dp[l][r]` — or LCS of `s` and `reversed(s)`.

**Ask first**
- Subsequence (gaps allowed), not substring? → Yes.

> 💡 **Idea.** `dp[l][r]` = longest palindromic subsequence in `s[l..r]`. Ends match → 2 + inside. Otherwise → max(drop left, drop right).

**Brute force:** try all subsequences, O(2ⁿ).

**Steps**
1. `dp[l][l] = 1`.
2. `s[l] == s[r]` → `dp[l][r] = dp[l + 1][r − 1] + 2`.
3. Otherwise `max(dp[l + 1][r], dp[l][r − 1])`.
4. Fill `l` from right to left (so row `l + 1` is ready), keeping one row and the diagonal.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **Fewest insertions to make it a palindrome (LC 1312)?** `n − LPS`.
- **Another way to get LPS?** LCS of `s` and `reversed(s)`.
- **O(n) space?** Fill `l` from right to left keeping one row.

In [35]:
def longest_palindrome_subseq(s):
    """Longest palindromic subsequence length."""
    n = len(s)
    # dp[l][r] = longest palindromic subsequence inside s[l..r]
    dp = [[0] * n for _ in range(n)]
    # 1-row version: dp = [0] * n
    for l in range(n - 1, -1, -1):
        # one character is a palindrome of length 1
        dp[l][l] = 1
        # 1-row version: prev_diag = 0; dp[l] = 1
        for r in range(l + 1, n):
            # matching ends wrap the inner answer
            if s[l] == s[r]:
                dp[l][r] = dp[l + 1][r - 1] + 2
            # drop the left end or the right end
            else:
                dp[l][r] = max(dp[l + 1][r], dp[l][r - 1])
            # 1-row version (replaces the if/else above):
            #   saved = dp[r]
            #   dp[r] = prev_diag + 2 if s[l] == s[r] else max(dp[r], dp[r - 1])
            #   prev_diag = saved
    return dp[0][n - 1]
    # 1-row version: return dp[-1]


assert longest_palindrome_subseq("bbbab") == 4
assert longest_palindrome_subseq("cbbd") == 2
print(longest_palindrome_subseq("bbbab"))

4


## 5.2 Two-string alignment

`dp[i][j]` over prefixes. Write the three moves — match/replace (diagonal), drop from `a` (up), drop from `b` (left) — and what each costs. Squeeze to one row by saving the diagonal before overwriting it.

### 5.2.1 [1143. Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest common subsequence of two strings.  
**Example.** `"abcde"` and `"ace"` → `3` (`"ace"`)  
**Spot it.** The canonical two-string DP.

**Ask first**
- Subsequence (not substring)? → Yes, gaps allowed.

> 💡 **Idea.** Look at the last characters. Equal → 1 + LCS of both without them. Different → the better of dropping one from either string.

**Brute force:** try all subsequences of one string, O(2ⁿ · m).

**Steps**
1. `dp[i][j]` = LCS of `a[:i]` and `b[:j]`; row and column 0 are 0.
2. Match → `dp[i − 1][j − 1] + 1`.
3. No match → `max(dp[i − 1][j], dp[i][j − 1])`.
4. Keep one row and the saved diagonal.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **Print the LCS itself?** Keep the full table and walk back from the bottom-right: diagonal on a match, else toward the bigger neighbour.
- **O(min(m, n)) space?** Two rows (or one row + a saved diagonal) — gives the length only.
- **Print it in linear space?** Hirschberg's divide and conquer: O(m·n) time, O(n) space.
- **Longest common *substring*?** `dp[i][j] = dp[i−1][j−1] + 1` on match, else 0; track the max.

In [36]:
def longest_common_subsequence(a, b):
    """LCS length."""
    # dp[i][j] = LCS of a[:i] and b[:j] (row 0 and column 0 stay 0: "" matches nothing)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    # 1-row version: dp = [0] * (len(b) + 1)
    for i in range(1, len(a) + 1):
        # 1-row version: diag = 0   (this is dp[i - 1][j - 1])
        for j in range(1, len(b) + 1):
            # match extends the diagonal, otherwise drop one character
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
            # 1-row version (replaces the if/else above):
            #   above = dp[j]
            #   dp[j] = diag + 1 if a[i - 1] == b[j - 1] else max(dp[j], dp[j - 1])
            #   diag = above
    return dp[len(a)][len(b)]
    # 1-row version: return dp[-1]


assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "def") == 0
print(longest_common_subsequence("abcde", "ace"))

3


### 5.2.2 [72. Edit Distance](https://leetcode.com/problems/edit-distance/) — ★★ · Medium

**Problem.** Fewest inserts, deletes and replaces to turn `a` into `b`.  
**Example.** `"horse"` → `"ros"` → `3`  
**Spot it.** Two-string DP with three operations.

**Ask first**
- All three operations cost 1? → Yes. Different costs just change the `+ 1`s.

> 💡 **Idea.** Last characters equal → no cost, recurse on both shorter. Otherwise 1 + min(insert, delete, replace) = 1 + min(left, up, diagonal).

**Brute force:** try all edit sequences recursively, exponential.

**Steps**
1. `dp[i][0] = i` (delete everything), `dp[0][j] = j` (insert everything).
2. Equal last characters → `dp[i − 1][j − 1]`.
3. Otherwise `1 + min(replace, delete, insert)`.
4. One row + saved diagonal; `dp[0]` becomes `i` at each new row.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **Return the operations?** Keep the full table and walk back from the corner, recording which move was used.
- **Different costs (replace = 2)?** Use the costs in the recurrence instead of 1.
- **Just check "one edit apart" (LC 161)?** One pass with two pointers, O(n) — no table needed.
- **O(n) space?** One row plus a saved diagonal value.

In [37]:
def min_distance(a, b):
    """Levenshtein distance."""
    # dp[i][j] = fewest edits to turn a[:i] into b[:j]
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    # row 0: turning "" into b[:j] needs j inserts
    for j in range(len(b) + 1):
        dp[0][j] = j
    # column 0: turning a[:i] into "" needs i deletes
    for i in range(len(a) + 1):
        dp[i][0] = i
    # 1-row version: dp = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        # 1-row version: diag, dp[0] = dp[0], i
        for j in range(1, len(b) + 1):
            # last characters equal: no edit needed
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            # replace (diagonal), delete (above) or insert (left)
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
            # 1-row version (replaces the if/else above):
            #   above = dp[j]
            #   dp[j] = diag if a[i - 1] == b[j - 1] else 1 + min(diag, above, dp[j - 1])
            #   diag = above
    return dp[len(a)][len(b)]
    # 1-row version: return dp[-1]


assert min_distance("horse", "ros") == 3
assert min_distance("intention", "execution") == 5
assert min_distance("", "abc") == 3
print(min_distance("horse", "ros"))

3


### 5.2.3 [97. Interleaving String](https://leetcode.com/problems/interleaving-string/) — ★★ · Medium

**Problem.** Is `s3` an interleaving of `s1` and `s2` (keeping each one's order)?  
**Example.** `s1 = "aabcc", s2 = "dbbca", s3 = "aadbbcbcac"` → `True`  
**Spot it.** `dp[i][j]` = `s3[:i+j]` is an interleaving of `s1[:i]` and `s2[:j]`.

**Ask first**
- Lengths don't add up? → Return False right away.

> 💡 **Idea.** `dp[i][j]` = can the first `i` chars of s1 and first `j` of s2 make the first `i + j` of s3? Come from the top if `s1[i−1]` matches, or from the left if `s2[j−1]` matches.

**Brute force:** try every way to pick the next char from s1 or s2, O(2^(m+n)).

**Steps**
1. Lengths must add up.
2. First row: prefixes of `s2` alone must match `s3`.
3. Each new row: column 0 uses `s1` alone.
4. `dp[j] = (dp[j] and s1[i−1] == s3[i+j−1]) or (dp[j−1] and s2[j−1] == s3[i+j−1])`.

**Cost:** Time O(mn) · Space O(n)

**Likely follow-ups**
- **O(len(s2)) extra memory (the official follow-up)?** Keep one row of length `len(s2) + 1` and update it in place.
- **Return one valid interleaving?** Walk back through the full table.

In [38]:
def is_interleave(s1, s2, s3):
    """True if s3 interleaves s1 and s2."""
    if len(s1) + len(s2) != len(s3):
        return False
    # dp[i][j] = True if s1[:i] and s2[:j] can interleave into s3[:i + j]
    dp = [[False] * (len(s2) + 1) for _ in range(len(s1) + 1)]
    dp[0][0] = True
    # 1-row version: dp = [True] + [False] * len(s2)
    # row 0: only characters from s2
    for j in range(1, len(s2) + 1):
        dp[0][j] = dp[0][j - 1] and s2[j - 1] == s3[j - 1]
        # 1-row version: dp[j] = dp[j - 1] and s2[j - 1] == s3[j - 1]
    for i in range(1, len(s1) + 1):
        # column 0: only characters from s1
        dp[i][0] = dp[i - 1][0] and s1[i - 1] == s3[i - 1]
        # 1-row version: dp[0] = dp[0] and s1[i - 1] == s3[i - 1]
        for j in range(1, len(s2) + 1):
            # last character came from s1 (above) or from s2 (left)
            from_s1 = dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]
            from_s2 = dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]
            dp[i][j] = from_s1 or from_s2
            # 1-row version: dp[j] = (dp[j] and s1[i - 1] == s3[i + j - 1]) or (
            # 1-row version:     dp[j - 1] and s2[j - 1] == s3[i + j - 1])
    return dp[len(s1)][len(s2)]
    # 1-row version: return dp[-1]


assert is_interleave("aabcc", "dbbca", "aadbbcbcac")
assert not is_interleave("aabcc", "dbbca", "aadbbbaccc")
assert is_interleave("", "", "")
print(is_interleave("aabcc", "dbbca", "aadbbcbcac"))

True


---
# 6) Interval and Game DP

**The gist.** Solve a problem on a range `[l, r]` by trying every way to split it and combining answers for smaller ranges. Game DP is the same with two players: your score minus your opponent's best on what's left.

> **Think of it like…** planning which building in a row comes down *last* — then the left and right blocks are separate jobs you already know how to price.

Fill by increasing length (or memoise) so smaller intervals are ready. Choosing the element handled **last** in the interval makes the two sides independent.

**Spot it:** "take from either end", "both players play optimally", "burst / merge / split a range".

**Classic bugs:** filling `dp[l][r]` before its smaller intervals are ready · scoring the game as "my total" instead of "my total − theirs".

## 6.1 Interval DP

Write `dp(l, r)` as "best result for `l..r`", then loop over the split or last action inside it. Start with `@cache` on a recursive version; turn it into a table only if needed.

### 6.1.1 [486. Predict the Winner](https://leetcode.com/problems/predict-the-winner/) — ★ · Medium

**Problem.** Players alternately take from either end of the array; can player 1 win (ties count)?  
**Example.** `[1, 5, 2]` → `False`; `[1, 5, 233, 7]` → `True`  
**Spot it.** Minimax as a score *difference*: `diff(l, r)` = current player's best margin.

**Ask first**
- Does a tie count as a win for player 1? → Yes, `≥ 0`.

> 💡 **Idea.** `diff(l, r)` = best (my score − your score) on `nums[l..r]` when it's my turn = max(`nums[l] − diff(l+1, r)`, `nums[r] − diff(l, r−1)`). Player 1 wins if `diff(0, n−1) ≥ 0`.

**Brute force:** explore the full game tree, O(2ⁿ).

**Steps**
1. `diff(l, l) = nums[l]`.
2. `diff(l, r) = max(nums[l] − diff(l + 1, r), nums[r] − diff(l, r − 1))`.
3. Player 1 wins iff `diff(0, n − 1) ≥ 0`.

**Cost:** Time O(n²) · Space O(n²)

**Likely follow-ups**
- **Even number of piles and odd total (LC 877)?** Player 1 always wins — they can take all even-index or all odd-index piles, whichever sum is bigger.
- **O(n) space?** Fill by increasing length and keep one array.
- **Return the moves?** Store the choice at each `(l, r)` and replay from `(0, n−1)`.

In [39]:
def predict_the_winner(nums):
    """True if player 1 can at least tie."""

    @cache
    def diff(l, r):
        """Best (my score - opponent score) on nums[l..r] for the player to move."""
        if l == r:
            return nums[l]
        # take an end, then the opponent plays optimally on the rest
        return max(nums[l] - diff(l + 1, r), nums[r] - diff(l, r - 1))

    return diff(0, len(nums) - 1) >= 0


assert not predict_the_winner([1, 5, 2])
assert predict_the_winner([1, 5, 233, 7])
print(predict_the_winner([1, 5, 233, 7]))

True


### 6.1.2 [877. Stone Game](https://leetcode.com/problems/stone-game/) — ★★ · Medium

**Problem.** Even number of piles, odd total; players take from either end. Does Alice win?  
**Example.** `[5, 3, 4, 5]` → `True`  
**Spot it.** Same margin DP — and a parity argument says the first player always wins.

**Ask first**
- Even number of piles and odd total? → Yes — that's what makes the trick work.

> 💡 **Idea.** The first player can always grab every even-indexed pile or every odd-indexed pile (whichever sum is bigger), so they always win → return True. The DP from Predict the Winner also works.

**Brute force:** minimax over every move, O(2ⁿ).

**Steps**
1. Reuse the margin DP to confirm (or just return `True`).

**Cost:** Time O(n²) with the DP, O(1) by the argument · Space O(n²) with the DP

**Likely follow-ups**
- **Prove the first player wins.** Colour piles alternately. Taking an end pile always leaves the opponent two ends of the *other* colour, so player 1 can take all of one colour.
- **What's the actual score difference?** Interval DP: `diff(l, r)` as in LC 486.
- **Odd number of piles or no parity guarantee?** The trick fails — use the DP.

In [40]:
def stone_game(piles):
    """True if Alice wins (always, for even pile counts)."""
    # the parity argument says True; the DP confirms it
    return predict_the_winner(piles)


assert stone_game([5, 3, 4, 5])
assert stone_game([3, 7, 2, 3])
print(stone_game([5, 3, 4, 5]))

True


---
# Summary

| Family | Recognise it by | State → transition | Cost |
|---|---|---|---|
| Fibonacci-style | ways / best up to i, adjacency ban | `dp[i]` from `dp[i−1]`, `dp[i−2]` | O(n), O(1) |
| Kadane | best contiguous subarray (sum, product, circular) | best ending at i: extend or restart | O(n) |
| Word / partition | segment a string, split a number | `dp[i]` over prefix lengths | O(n·L) |
| LIS | increasing / chained subsequence | patience `tails` + binary search | O(n log n) |
| 0/1 knapsack | subset sum, partition, target sum | capacity loop **downwards** | O(n·W) |
| Unbounded knapsack | coins, squares | capacity loop **upwards**; loop order = combos vs sequences | O(n·W) |
| State machine | stock buy/sell with limits | one variable per state, yesterday's values | O(n·k) |
| Grid | right/down paths | one row array, in-place update | O(mn), O(n) |
| Memo DFS on DAG | longest increasing path | `@cache` on cell | O(mn) |
| Palindromes | substrings / subsequences | expand around centres; `dp[l][r]` | O(n²) |
| Two strings | LCS, edit distance, interleave | `dp[i][j]` over prefixes | O(mn) |
| Interval / game | take from either end, both play optimally | `dp(l, r)` = best score difference on `[l, r]` | O(n²) |